# Dynamical Lie Algebra of a Three-Qubit Ising Chain

## Background

The *dynamical Lie algebra* (DLA) generated by Hermitian operators $\{H_j\}$ is the real vector space spanned by the anti-Hermitian generators $\{-iH_j\}$ and all their nested commutators. The unitaries a circuit built from these generators can reach form the group this algebra generates, so its dimension counts the independent directions the circuit can steer in. Chapter 1 computes it for the two-qubit transverse-field Ising model, where it is six-dimensional.

## Exercise

Consider the open three-qubit transverse-field Ising chain, with generators

$$
Z_1,\quad Z_2,\quad Z_3,\quad X_1X_2,\quad X_2X_3,
$$

where $Z_1 = \sigma_z \otimes I \otimes I$, $X_1X_2 = \sigma_x \otimes \sigma_x \otimes I$, and so on.

**(a)** Show that two Pauli strings $P$, $Q$ either commute or anticommute, and that when they anticommute, $[-iP, -iQ]$ is a real multiple of $-iR$ for a single Pauli string $R$ proportional to $PQ$. New elements of the DLA are therefore new Pauli strings.

**(b)** Starting from the five generators, take commutators until no new strings appear. Show that the DLA is spanned by fifteen strings and has $\dim\mathfrak{g} = 15$.

**(c)** Show that the parity $Z_1Z_2Z_3$ commutes with all five generators. As in the two-qubit example, this confines $\mathfrak{g}$ to operators that are block diagonal and blockwise traceless in the two four-dimensional parity sectors. Compare the dimension of that space with $\dim\mathfrak{g}$ and with $\dim\mathfrak{su}(8) = 63$. Does parity alone account for the restriction?

## Solution

### Part (a)

On each site two Pauli matrices either commute or anticommute, so two strings commute when they anticommute on an even number of sites and anticommute otherwise. If $PQ = -QP$, then

$$
[-iP, -iQ] = -(PQ - QP) = -2PQ .
$$

The product $PQ$ is a phase times a Pauli string $R$, by $\sigma_j\sigma_k = \delta_{jk}I + i\epsilon_{jkl}\sigma_l$ on each site, and it is anti-Hermitian because $(PQ)^\dagger = QP = -PQ$, so the phase is $\pm i$ and the commutator is $\mp 2iR$.

### Part (b)

The first commutators act within each bond: $[-iZ_1, -iX_1X_2] = -2i\,Y_1X_2$, then $Z_2$ gives $X_1Y_2$, and $Z_1$ acting on that gives $Y_1Y_2$; the bond $(2,3)$ gives $Y_2X_3$, $X_2Y_3$ and $Y_2Y_3$ in the same way. The two bond generators commute, $[X_1X_2, X_2X_3] = 0$, but the new strings join the bonds through site 2:

$$
[-iX_1Y_2,\, -iX_2X_3] = 2i\,X_1Z_2X_3 ,
$$

and commutators with $Z_1$ and $Z_3$ turn the end $X$'s into $Y$'s. The DLA is spanned by $-i$ times

$$
Z_1,\ Z_2,\ Z_3;\qquad A_1B_2,\ A_2B_3,\ A_1Z_2B_3 \qquad (A, B \in \{X, Y\}),
$$

three plus $3\times4$ strings, and the commutator of any two of them that anticommute is again on the list. Hence $\dim\mathfrak{g} = 15$.

### Part (c)

$Z_1Z_2Z_3$ commutes with each $Z_i$, and with $X_1X_2$ and $X_2X_3$ because each of those anticommutes with it on exactly two sites. Blockwise traceless operators on two four-dimensional sectors form $\mathfrak{su}(4)\oplus\mathfrak{su}(4)$, of dimension $2\times15 = 30$. The DLA has dimension 15, half of what parity allows, and far below 63. Parity therefore does not account for the whole restriction. The remaining constraint is the free-fermion structure of the Ising chain, which the Jordan–Wigner mapping of Chapter 4 makes explicit; for an open chain of $N$ sites the DLA has dimension $N(2N-1)$, which is 6, 15 and 28 for $N = 2, 3, 4$, and grows only quadratically while $\dim\mathfrak{su}(2^N) = 4^N - 1$ grows exponentially.

---
## Numerical Verification

In [1]:
import numpy as np
import itertools

I2 = np.eye(2, dtype=complex)
PAULI = {'I': I2,
         'X': np.array([[0, 1], [1, 0]], dtype=complex),
         'Y': np.array([[0, -1j], [1j, 0]]),
         'Z': np.array([[1, 0], [0, -1]], dtype=complex)}

def string(s):
    """Pauli string, e.g. 'XZY' = sigma_x (x) sigma_z (x) sigma_y."""
    out = np.array([[1]], dtype=complex)
    for c in s:
        out = np.kron(out, PAULI[c])
    return out

def rank(ops):
    return np.linalg.matrix_rank(np.array([o.flatten() for o in ops]), tol=1e-9)

def dla(generators):
    """Close {-i H_j} under commutators; return a list of basis elements."""
    basis = [-1j * H for H in generators]
    changed = True
    while changed:
        changed = False
        for A, B in itertools.product(list(basis), list(basis)):
            C = A @ B - B @ A
            if np.linalg.norm(C) > 1e-9 and rank(basis + [C]) > len(basis):
                basis.append(C)
                changed = True
    return basis

def ising_generators(N):
    """Open transverse-field Ising chain: Z_i and X_i X_{i+1}."""
    zs = ['I' * i + 'Z' + 'I' * (N - i - 1) for i in range(N)]
    xx = ['I' * i + 'XX' + 'I' * (N - i - 2) for i in range(N - 1)]
    return [string(s) for s in zs + xx]

# (b) three-qubit chain
g = dla(ising_generators(3))
print("dim DLA (N = 3):", len(g))

claimed = ['ZII', 'IZI', 'IIZ',
           'XXI', 'XYI', 'YXI', 'YYI',
           'IXX', 'IXY', 'IYX', 'IYY',
           'XZX', 'XZY', 'YZX', 'YZY']
C = [-1j * string(s) for s in claimed]
print("the 15 listed strings are independent:", rank(C) == 15)
print("and span the DLA:", rank(C + g) == 15)

# first commutators quoted in the solution
comm = lambda A, B: A @ B - B @ A
print("[-iZ1, -iX1X2] = -2i Y1X2:", np.allclose(comm(-1j*string('ZII'), -1j*string('XXI')), -2j*string('YXI')))
print("[X1X2, X2X3] = 0:", np.allclose(comm(string('XXI'), string('IXX')), 0))
print("[-iX1Y2, -iX2X3] = 2i X1Z2X3:", np.allclose(comm(-1j*string('XYI'), -1j*string('IXX')), 2j*string('XZX')))

# (c) parity
P = string('ZZZ')
print("Z1Z2Z3 commutes with every generator:", all(np.allclose(P @ H, H @ P) for H in ising_generators(3)))
print("-i Z1Z2Z3 lies in the DLA:", rank(C + [-1j * P]) == 15)
print("parity-allowed dimension su(4)+su(4) = 30; dim su(8) = 63")

# growth with N
for N in (2, 3, 4):
    print(f"N = {N}: dim DLA = {len(dla(ising_generators(N)))},  N(2N-1) = {N*(2*N-1)},  4^N - 1 = {4**N - 1}")


dim DLA (N = 3): 15
the 15 listed strings are independent: True
and span the DLA: True
[-iZ1, -iX1X2] = -2i Y1X2: True
[X1X2, X2X3] = 0: True
[-iX1Y2, -iX2X3] = 2i X1Z2X3: True
Z1Z2Z3 commutes with every generator: True
-i Z1Z2Z3 lies in the DLA: False
parity-allowed dimension su(4)+su(4) = 30; dim su(8) = 63
N = 2: dim DLA = 6,  N(2N-1) = 6,  4^N - 1 = 15
N = 3: dim DLA = 15,  N(2N-1) = 15,  4^N - 1 = 63


N = 4: dim DLA = 28,  N(2N-1) = 28,  4^N - 1 = 255


---
## Takeaway

The open three-qubit Ising chain generates a fifteen-dimensional algebra inside the sixty-three-dimensional $\mathfrak{su}(8)$. Parity accounts for only part of the restriction, allowing thirty dimensions; the rest comes from the free-fermion structure of the chain, and the dimension grows as $N(2N-1)$.

## References

- Larocca et al., *Group-Invariant Quantum Machine Learning*, PRX Quantum (2022).
- Goodman and Wallach, *Symmetry, Representations, and Invariants*, Springer (2009).
- Płodzień, *Information Scrambling, Quantum Chaos, and Haar-Random States*, lecture notes (2025). [arXiv:2511.14397](https://arxiv.org/abs/2511.14397)